In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:

%pip install --quiet geopandas fiona

%pip install rasterio

%pip install pystac



In [3]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os
import csv

import json
import xml.etree.ElementTree as ET
import datetime
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import shape, mapping, MultiPolygon, Polygon,box
from IPython.display import Image, display


import numpy as np
import pystac
from pystac.extensions.table import TableExtension
from pystac import CatalogType
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


### **Used ijson python package for reading the large GEOJSON files and get the metadata **

In [4]:
pip install ijson

In [5]:
import datetime
from pystac import Item
import pandas as pd
import geopandas as gpd
import os
import ijson
import json
from shapely.geometry import shape, mapping

In [6]:
# #geojson_filepath = '/content/drive/MyDrive/Pan_india/pan_india_drainage_lines.geojson'
# geojson_filepath = '/content/drive/MyDrive/Pan_india/Microwatershed_boundries_v2.geojson'

# geojson_filepath = "/content/drive/MyDrive/PAN_india_vector_layerstest.csv"

geojson_filepath = "/content/drive/MyDrive/PAN_india_layers_updated.csv"

STAC_SAVE_DIR = "/content/drive/MyDrive/STAC_spec_vector_test"
SUB_COLLECTIONS = {}

In [7]:
df = pd.read_csv(geojson_filepath)

print("DataFrame loaded successfully")
print(df.head())

DataFrame loaded successfully
                Layer Name                                         asset link  \
0          Drainage layers                                                NaN   
1           Drainage Lines  /content/drive/MyDrive/Pan_india/pan_india_dra...   
2  Hydrological boundaries                                                NaN   
3                Sub-basin  /content/drive/MyDrive/Pan_india/CWC_subbasin_...   
4                Watershed  /content/drive/MyDrive/Pan_india/Watershed_pan...   

                                          drive_link  
0                                                NaN  
1  https://drive.google.com/file/d/1jNUbJj41qUJZj...  
2                                                NaN  
3  https://drive.google.com/file/d/1-q-angbycEG9k...  
4  https://drive.google.com/file/d/1eKZLpBWQDbT4A...  


In [8]:
try:
    # Define COLUMN_DESC_DF globally for use in run_stac_generation
    COLUMN_DESC_DF = pd.read_csv('/content/drive/MyDrive/column_description_9dec.csv')
    print("Column descriptions loaded successfully.")
except FileNotFoundError:
    print("WARNING: 'column_descriptions.csv' not found. Table extension will be empty.")
    COLUMN_DESC_DF = pd.DataFrame({'layer_name': [], 'column_name': [], 'column_name_description': []})

Column descriptions loaded successfully.


In [10]:
def create_root_and_collection():
    root_catalog = pystac.Catalog(
        id="PANindia",
        title="STAC Catalog for PAN India Vector Layers",
        description="Root catalog for PAN India Vector assets and their metadata."
    )

    panindia_collection = pystac.Collection(
        id="panindia-vector-layers",
        title="CoRE stack Pan India Vector Layers",
        description="A collection of various vector layers for Pan India.",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[68, 8, 98, 37]]),
            temporal=pystac.TemporalExtent([[datetime.datetime(2005, 1, 1), datetime.datetime(2024, 12, 31)]])
        ),
        license="CC-BY-4.0",
        providers=[
            pystac.Provider(
                name="CoREstack",
                roles=[
                    pystac.ProviderRole.PRODUCER,
                    pystac.ProviderRole.PROCESSOR,
                    pystac.ProviderRole.HOST
                ],
                url="https://core-stack.org/"
            )
        ],
        keywords=["social-ecological", "sustainability", "CoRE stack"]
    )

    root_catalog.add_child(panindia_collection)
    return root_catalog, panindia_collection


In [11]:
def create_sub_collection(title, parent_collection):

    title_str = str(title) if title is not None else "unknown"
    collection_id = title_str.lower().replace(' ', '-').replace(':', '').replace('/', '-')

    description = f"STAC collection {title_str} for Pan India."

    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection: {title_str}")
        sub_collection = pystac.Collection(
            id=collection_id,
            title=title_str,
            description=description,
            extent=parent_collection.extent,
            license=parent_collection.license,
            providers=parent_collection.providers
        )


        SUB_COLLECTIONS[collection_id] = sub_collection
    return SUB_COLLECTIONS[collection_id]


In [12]:
def get_feature_geometry(geojson_filepath):

    with open(geojson_filepath, 'r') as f:
        for item in ijson.items(f, 'features.item', use_float=True):
            geom = item.get("geometry")
            props = item.get("properties", {})
            shapely_geom = shape(geom)
            minx, miny, maxx, maxy = shapely_geom.bounds
            return geom, [minx, miny, maxx, maxy]

In [13]:

def get_first_feature_properties(geojson_filepath):

    with open(geojson_filepath, 'r') as f:
        for item in ijson.items(f, 'features.item.properties', use_float=True):
            properties_dict = {
                key: {
                    "value": value,
                    "type": type(value).__name__
                }
                for key, value in item.items()
            }
            break

    return {
        "properties": properties_dict
    }


In [14]:

def load_layer_descriptions(csv_path="/content/drive/MyDrive/layer_descriptions.csv"):
    desc_map = {}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:

            norm_key = (
                row["layer_name"]
                .lower()
                .replace(" ", "-")
                .replace(":", "")
                .replace("/", "-")
                .strip()
            )
            desc_map[norm_key] = row["layer_description"].strip()
    return desc_map

LAYER_DESCRIPTIONS = load_layer_descriptions("/content/drive/MyDrive/layer_descriptions.csv")


In [15]:
# def generate_vector_stac(layer_name, file_path, column_desc_df, drive_link=None):
#     print(f"Processing vector layer: {layer_name}")

#     #item_id = layer_name.replace(" ", "_").replace("/", "_").replace("-", "_").lower()

#     item_id = (layer_name.lower().replace(" ", "-").replace(":", "").replace("/", "-").strip())

#     item_description = LAYER_DESCRIPTIONS.get(item_id, layer_name)

#     footprint, bbox = get_feature_geometry(file_path)
#     properties_dict = get_first_feature_properties(file_path)["properties"]

#     vector_gdf_dtypes = pd.DataFrame(
#         [(key, value['type']) for key, value in properties_dict.items()],
#         columns=['column_name', 'column_dtype']
#     )

#     try:
#         vector_item = pystac.Item(
#             id=item_id,
#             geometry=footprint,
#             bbox=bbox,
#             datetime=datetime.datetime.now(datetime.timezone.utc),
#             properties={"description": item_description}
#         )

#         proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
#         proj_ext.epsg = 4326

#         layer_filter_name = item_id
#         vector_desc_filtered_df = column_desc_df[column_desc_df['layer_name'] == layer_filter_name]

#         vector_merged_df = vector_gdf_dtypes.merge(
#             vector_desc_filtered_df[['column_name', 'column_name_description']],
#             on='column_name',
#             how='left'
#         ).fillna('')

#         vector_merged_df.rename(columns={'column_name_description':'column_description'}, inplace=True)
#         print(f"Schema merged for {layer_name}. Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} descriptions.")


#         table_ext = TableExtension.ext(vector_item, add_if_missing=True)
#         table_ext.columns = [
#             {
#                 "name": row['column_name'],
#                 "type": str(row['column_dtype']),
#                 "description" : row['column_description']
#             }
#             for ind, row in vector_merged_df.iterrows()
#         ]


#         if drive_link:
#             vector_item.add_asset("drive-link", pystac.Asset(
#                 href=drive_link,
#                 # href='https://zenodo.org/records/17827623',
#                 media_type=pystac.MediaType.TEXT,
#                 roles=["source"],
#                 title="Vector File"
#             ))
#         else:
#             print(f"No Drive link found for {layer_name}, skipping asset attachment.")

#         return vector_item

#     except Exception as e:
#         print(f" Error generating STAC Item for {layer_name}: {e}")
#         return None

In [16]:
def generate_vector_stac(layer_name, file_path, column_desc_df, drive_link=None):
    print(f"Processing vector layer: {layer_name}")

    item_id = (
        layer_name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .strip()
    )


    description_id = (
        layer_name.lower()
        .replace(" ", "-")
        .replace(":", "")
        .replace("/", "-")
        .strip()
    )


    item_description = LAYER_DESCRIPTIONS.get(description_id, layer_name)


    footprint, bbox = get_feature_geometry(file_path)
    properties_dict = get_first_feature_properties(file_path)["properties"]


    vector_gdf_dtypes = pd.DataFrame(
        [(key, value['type']) for key, value in properties_dict.items()],
        columns=['column_name', 'column_dtype']
    )

    try:

        vector_item = pystac.Item(
            id=item_id,
            geometry=footprint,
            bbox=bbox,
            datetime=datetime.datetime.now(datetime.timezone.utc),
            properties={"description": item_description}
        )

        # Projection extension
        proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
        proj_ext.epsg = 4326


        csv_layer_normalized = (
            column_desc_df["layer_name"]
            .str.lower()
            .str.replace(" ", "_")
            .str.replace("/", "_")
            .str.replace("-", "_")
            .str.strip()
        )


        vector_desc_filtered_df = column_desc_df[csv_layer_normalized == item_id]


        vector_merged_df = vector_gdf_dtypes.merge(
            vector_desc_filtered_df[['column_name', 'column_name_description']],
            on='column_name',
            how='left'
        ).fillna('')

        vector_merged_df.rename(
            columns={'column_name_description': 'column_description'},
            inplace=True
        )

        print(
            f"Schema merged for {layer_name}. "
            f"Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} column descriptions."
        )

        # Table extension
        table_ext = TableExtension.ext(vector_item, add_if_missing=True)
        table_ext.columns = [
            {
                "name": row['column_name'],
                "type": str(row['column_dtype']),
                "description": row['column_description']
            }
            for idx, row in vector_merged_df.iterrows()
        ]


        if drive_link:
            vector_item.add_asset(
                "drive-link",
                pystac.Asset(
                    href=drive_link,
                    media_type=pystac.MediaType.TEXT,
                    roles=["source"],
                    title="Vector File"
                )
            )
        else:
            print(f"No Drive link found for {layer_name}, skipping asset attachment.")

        return vector_item

    except Exception as e:
        print(f"Error generating STAC Item for {layer_name}: {e}")
        return None


In [17]:
# def generate_vector_stac(layer_name, file_path, column_desc_df, drive_link=None):
#     print(f"Processing vector layer: {layer_name}")

#     item_id = layer_name.replace(" ", "_").replace("/", "_").replace("-", "_").lower()

#     description_id = (layer_name.lower().replace(" ", "-").replace(":", "").replace("/", "-").strip())

#     item_description = LAYER_DESCRIPTIONS.get(description_id, layer_name)

#     footprint, bbox = get_feature_geometry(file_path)
#     properties_dict = get_first_feature_properties(file_path)["properties"]

#     vector_gdf_dtypes = pd.DataFrame(
#         [(key, value['type']) for key, value in properties_dict.items()],
#         columns=['column_name', 'column_dtype']
#     )

#     try:
#         vector_item = pystac.Item(
#             id=item_id,
#             geometry=footprint,
#             bbox=bbox,
#             datetime=datetime.datetime.now(datetime.timezone.utc),
#             properties={"description": item_description}
#         )

#         proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
#         proj_ext.epsg = 4326

#         layer_filter_name = item_id
#         vector_desc_filtered_df = column_desc_df[column_desc_df['layer_name'] == layer_filter_name]

#         vector_merged_df = vector_gdf_dtypes.merge(
#             vector_desc_filtered_df[['column_name', 'column_name_description']],
#             on='column_name',
#             how='left'
#         ).fillna('')

#         vector_merged_df.rename(columns={'column_name_description':'column_description'}, inplace=True)
#         print(f"Schema merged for {layer_name}. Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} descriptions.")


#         table_ext = TableExtension.ext(vector_item, add_if_missing=True)
#         table_ext.columns = [
#             {
#                 "name": row['column_name'],
#                 "type": str(row['column_dtype']),
#                 "description" : row['column_description']
#             }
#             for ind, row in vector_merged_df.iterrows()
#         ]


#         if drive_link:
#             vector_item.add_asset("drive-link", pystac.Asset(
#                 href=drive_link,
#                 media_type=pystac.MediaType.TEXT,
#                 roles=["source"],
#                 title="Vector File"
#             ))
#         else:
#             print(f"No Drive link found for {layer_name}, skipping asset attachment.")

#         return vector_item

#     except Exception as e:
#         print(f" Error generating STAC Item for {layer_name}: {e}")
#         return None

In [18]:
def run_stac_generation(df, panindia_collection, root_catalog, column_desc_df):
    generated_items_count = 0
    current_collection = panindia_collection


    search_terms_filepath = ['Vector file path', 'FilePath', 'URL', 'Link']
    file_path_col = None
    for term in search_terms_filepath:
        matches = [col for col in df.columns if term.lower() in col.lower()]
        if matches:
            file_path_col = matches[0]
            break

    drive_link_col_matches = [col for col in df.columns if 'drive_link'.lower() in col.lower()]
    drive_link_col = drive_link_col_matches[0] if drive_link_col_matches else None

    if file_path_col is None:
        print("Fatal Error: Could not find a suitable column for local file paths.")
        print(f"Available columns: {list(df.columns)}")
        return

    print(f"File Path Column: {file_path_col}")
    print(f"Drive Link Column: {drive_link_col if drive_link_col else 'None'}")

    for idx, row in df.iterrows():
        layer_name = row["Layer Name"]
        file_path = row[file_path_col]
        drive_link = (
            row[drive_link_col]
            if drive_link_col and not pd.isna(row[drive_link_col]) and str(row[drive_link_col]).strip() != ''
            else None
        )

        if pd.isna(file_path) or str(file_path).strip() == '':

            current_collection = create_sub_collection(layer_name, panindia_collection)
            panindia_collection.add_child(current_collection)
            print(f"Created sub-collection: {layer_name}")
            continue

        vector_item = generate_vector_stac(layer_name, file_path, column_desc_df, drive_link)

        if vector_item:
            current_collection.add_item(vector_item)
            generated_items_count += 1
            print(f"Added Item: {vector_item.id} to Collection: {current_collection.id}")

    save_catalog(root_catalog, generated_items_count)

In [19]:
def save_catalog(root_catalog, count):
    if count > 0:
        root_catalog.normalize_hrefs(STAC_SAVE_DIR)
        root_catalog.save(catalog_type=CatalogType.SELF_CONTAINED)
        print(f"STAC catalog saved")
        print(f"Total items generated: {count}")
    else:
        print("No items were generated.")


root_catalog, panindia_collection = create_root_and_collection()
run_stac_generation(df, panindia_collection, root_catalog, COLUMN_DESC_DF)

File Path Column: asset link
Drive Link Column: drive_link
Creating new Sub-Collection: Drainage layers
Created sub-collection: Drainage layers
Processing vector layer: Drainage Lines
Schema merged for Drainage Lines. Found 7 column descriptions.
Added Item: drainage_lines to Collection: drainage-layers
Creating new Sub-Collection: Hydrological boundaries
Created sub-collection: Hydrological boundaries
Processing vector layer: Sub-basin
Schema merged for Sub-basin. Found 4 column descriptions.
Added Item: sub_basin to Collection: hydrological-boundaries
Processing vector layer: Watershed
Schema merged for Watershed. Found 8 column descriptions.
Added Item: watershed to Collection: hydrological-boundaries
Processing vector layer: Microwatershed boundaries v1
Schema merged for Microwatershed boundaries v1. Found 0 column descriptions.
Added Item: microwatershed_boundaries_v1 to Collection: hydrological-boundaries
Processing vector layer: Microwatershed boundaries v2
Schema merged for Mic